<a href="https://colab.research.google.com/github/closes/earth_observation_samos/blob/main/Data_skills_for_satellite_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Option 1: download the data within python.

For these exercises, if you want to download the data directly within python, then you will start by installing the `copernicusmarine` toolbox again so that you can access the data:

In [ ]:
!pip install copernicusmarine

In [ ]:
import copernicusmarine as cm

And connect to your account:

In [ ]:
cm.login(username="enter_your_username")

# Option 2: download the data from the Copernicus Marine website

If you prefer to download the data using the graphical interface or the subsetting tool, then you can download the data directly from the [Copernicus Marine website](https://marine.copernicus.eu/), and then load it into python afterwards.

## A. Calculating seasonal cycles and anomalies

Many variables have a seasonal cycle that is much stronger than the interannual changes. Since the seasonal cycle is predictable, normally we are much less interested in this than in changes at other time scales, and it is often useful to remove it from the data. This gives us a time series of anomalies, where it is much easier to see these other time scales.

Let's look at an example using SST again.

We'll use the same data as for previous days: the [ESA SST CCI and C3S reprocessed sea surface temperature analyses](https://data.marine.copernicus.eu/product/SST_GLO_SST_L4_REP_OBSERVATIONS_010_024/description) product.

In the cell below, load:
- the `sea surface temperature` variable from the dataset called "Copernicus Climate Change Service"
- over the region: 35-35.2°E, 35-35.2°S
- and the period: 1st January 1995 - 31st December 2005


In [ ]:
ds = cm.open_dataset(
    dataset_id= ,# string
    variables= , # list of strings
    start_datetime= , #string
    end_datetime= , #string
    minimum_longitude= , # float
    maximum_longitude= , # float
    minimum_latitude= , # float
    maximum_latitude= , # float
)

In [ ]:
ds

Your data set should contain 4018 time points, and 4x4 spatial points.

In the cell below, average over the latitude and longitude dimensions (you can use the `ds.mean()` method again), to obtain 1 value per time step. Plot the time series. What do you see?

In [ ]:
# enter your code here

The seasonal cycle represents the expected value of the temperature (or other variable) for each day of the year.

To calculate the seasonal cycle, we need to calculate the average value for each day of the year.

- Our time series contains the years 1995-2005. We have one value for each of the 365 days of the year, every year.
- The seasonal cycle will contain 365 points, one for each day of the year
- The first value of the seasonal cycle will give us the expected value on the 1st January. To calculate it, we need to average our values of the temperature for: 1st January 1995, 1st January 1996, 1st January 1997, ..., 1st January 2015
- We then repeat this calculation for the 2nd January to get the 2nd value, for the 3rd January to get the 3rd value... etc.

Let's see how to calculate the value for 1st January:

In [ ]:
# first let's make the spatial mean:
ds_mean = ds.mean(dim=('latitude','longitude')).analysed_sst

# we need to know the day of the year and the month for each
# time in the data set
# we can calculate this easily using xarray:
day = ds.time.dt.day.values
month = ds.time.dt.month.values

print('First 10 values in day array: ', day[:10])
print('First 10 values in month array: ', month[:10])

In [ ]:
# Now we will use this information to select the data points that we
# will average. Let's find the indices of all the points where
# the date is 1st January. This corresponds to points where:
# the day is 1, and the month is 1.
import numpy as np
index = np.where((day == 1) & (month == 1))[0]
print('First 10 indices: ', index)
print('Time values for the selected indices:',ds.isel(time=index).time.values)
print('')

In [ ]:
# Now we can get the temperature values on those days:
jan_01 = ds_mean.isel(time=index)
print('Temperature values for the selected indices:',jan_01.values)
print('')

In [ ]:
# and finally we can calculate the mean and the anomalies:
tmean_0101 = jan_01.mean(dim='time').values
tanom = jan_01.values - tmean_0101
print('Mean temperature on 1st January: ', tmean_0101)
print('Anomaly temperature on 1st January: ', tanom)

In [ ]:
# and let's plot some histograms to look at the values:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,3))
plt.subplot(121)
plt.hist(jan_01.values, bins=5, edgecolor='k')
plt.xlabel('Temperature [°K]')
# we'll add a red line to show the mean value
plt.axvline(tmean_0101, color='r')
plt.title('Temperature on 1st January')
plt.subplot(122)
plt.hist(tanom, bins=5, edgecolor='k')
plt.xlabel('Temperature anomaly [°K]')
plt.title('Temperature anomaly on 1st January')
plt.show()

In the cell below, try to calculate the seasonal cycle by calculating the average temperature for every day of the year (1st January, 2nd January, ..., 31st December). You will need to store your averages in a new array. You should have an array with 365 values of average temperature at the end of the calculation.

In [ ]:
# enter your code here

Now plot your seasonal cycle. For your x-axis data, you can use `np.arange(1,367)` to generate 366 points to represent the days of the year (including 29 Feb).

In [ ]:
# enter your code here

Finally, let's calculate the anomalies:
- Make a copy of your original mean temperature array, containing 4018 points
- Use the same method as above to identify the points in the array which correspond to each day of the year
- Subtract your seasonal cycle value from those points to give the anomalies
- Store the anomalies in your copy of the temperature array

You can refer to the worked example above again to help you compute the anomalies.

In [ ]:
# enter your code here

And finally, plot the anomaly time series. Do you notice any features that were difficult to see in the original time series?

In [ ]:
# enter your code here

## B. Fitting linear and non-linear trends

We might want to characterise the long term behaviour of a time series either using a straight line (linear trend) or some form of curve (non-linear trend).

The straight line will allow us to express a rate of change for our chosen variable. There is no strict equivalent of this for the non-linear case, but this non-linear approach can still be useful if we are trying to model the long-term behaviour of the time series. We might want to do this either to filter out the effects of short term variations, or to focus on them more clearly.

Let's look at the two approaches. We'll start with the linear trend.

For the following exercises, we'll reuse the temperature anomaly time series that you calculated in part A.

### B1. Linear trends

To calculate a linear trend, we want to fit the line: $y=ax+b$. We need to estimate the coefficients (numbers) $a$ and $b$.

We can do this in python using `numpy`'s `polyfit` method.
Let's look at the documentation:

In [ ]:
import numpy as np
np.polyfit?

We can see in the "parameters" section of the documentation that we need to supply our input data, x and y, and also the degree of the polynomial. For a straight line, this corresponds to setting `deg=1`.

In the "returns" section of the documentation, we can see that the function will return un an array, p, which contains the coefficients. So with `deg=1`, the array p will return us the values of $a$ and $b$. It returns $a$ as the first element, and $b$ as the second.

Let's try an example again using some made-up data:

In [ ]:
fake_time_data = np.arange(1,10,0.1)
fake_y_data = 1*fake_time_data + np.random.randn(len(fake_time_data))
# and let's plot this made up data
plt.plot(fake_time_data,fake_y_data)

# now we'll calculate the trend:
p = np.polyfit(fake_time_data,fake_y_data,deg=1)
print('The coefficients are: ', p)

# and plot the straight line on the graph:
trend_line = p[1] + p[0]*fake_time_data
plt.plot(fake_time_data,trend_line,'r-');

In the cell below, calculate the linear trend for your temperature anomaly time series from part A. Plot the time series and your calculated trend.

In [ ]:
# enter your code here

What is the value of the trend? What are its physical units?

### B2. Non-linear trends

Non-linear trends are any long-term behaviour that is not a straight line. This represents a lot of possibilities! The behaviour could be quadratic, cubic, exponential, or something else completely.

If we have no idea what function is best to describe the long-term behaviour, a fairly common approach is to use a polynomial fit to the data. In python, we can use the same `polyfit` function that we used above to achieve this.

The difference lies in the `deg` parameter for the degree of the polynomial. With:
- `deg=1`, we fit: $y=ax+b$. This is a straight line. The array p will contain 2 values, $a$ and $b$.
- `deg=2`, we fit: $y=ax²+bx+c$. This is a parabola (bowl shape). The array p will contain 3 values, $a$, $b$ and $c$.
- `deg=3`, we fit: $y=ax^3+bx^2+cx+d$. This is a cubic function. The array p will contain 4 values, $a$, $b$, $c$ and $d$.

and so on. We can increase the degree as much as we like and the function will become more and more complicated.

It can be tiresome to calculate the trend line manually using the values in p for higher degree polynomials. There is a function that can do this calculation for us, called `polyval`. You can check the documentation, like we did for polyfit, to see how it works.

Let's make some tests on made-up data again:


In [ ]:
fake_time_data = np.arange(1,10,0.1)
fake_y_data = 0.8*fake_time_data + \
              5*np.random.randn(len(fake_time_data)) + \
              5e-3*fake_time_data**3
# and let's plot this made up data
plt.plot(fake_time_data,fake_y_data)

# now we'll calculate the non-linear trend for a 8th order polynomial:
p = np.polyfit(fake_time_data,fake_y_data,deg=8)
print('The coefficients are: ', p)

# now let's calculate the fitted line using polyfit:
trend_line = np.polyval(p,fake_time_data)

# and plot the trend on the graph:
plt.plot(fake_time_data,trend_line,'r-');
plt.title('Fake data and fitted 8th degree polynomial')
plt.grid()

# make a second plot showing the anomalies around the trend:
plt.figure()
plt.plot(fake_time_data,fake_y_data-trend_line)
plt.grid()
plt.title('Anomalies around the fitted line');

You might see some unrealistic-looking wiggles in the fitted line above, particularly near the beginning and end of the time series. This is a common problem with higher order polynomials, and it is usually best to use the simplest fit that gives you a reasonable description of the data.

Try fitting different degrees of polynomial, from 1 to 5, to your temperature anomaly data from part A.

For each fit, make 2 plots showing:
- your data, with the estimated trend plotted over the top
- the anomalies after you subtract the estimated trend from your data

In [ ]:
# enter your code here

How would you assess the different fits to your data? Do you think it is useful to use a non-linear trend in this case? Or does the straight line do a good enough job?

## C. Interpolating and filling in missing or "bad" data

Interpolation allows us to estimate values that are missing, or considered to be unreliable, by using the data of nearby points. It can also allow us to switch from one grid to another one if we are using multiple data sets and want to compare data at the same point.

Lots of different methods of interpolation exist, some of which are very sophisticated. We will look at some simple methods today. These are generally much too simple for use in applications such as filling the gaps in cloud data. However, the basic principles remain the same.

We'll look at 2 basic methods today: the nearest neighbours method and linear interpolation.

We'll consider the case of trying to transform one gridded product onto another grid in the first instance.

Let's download the same SST and ocean colour products that we used yesterday. We'll just get the data for 1 day to start with:

In [ ]:
ds_colour = cm.open_dataset(
    dataset_id= "cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D",# string
    variables= ["CHL"], # list of strings
    start_datetime= "2023-04-01", #string
    end_datetime= "2023-04-01", #string
    minimum_longitude= 16.5, # float
    maximum_longitude= 20, # float
    minimum_latitude= -36, # float
    maximum_latitude= -30, # float
)

ds_sst = cm.open_dataset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ['analysed_sst'], # list of strings
    start_datetime= "2023-04-01", #string
    end_datetime= "2023-04-01", #string
    minimum_longitude= 16.5, # float
    maximum_longitude= 20, # float
    minimum_latitude= -36, # float
    maximum_latitude= -30, # float
)

Inspect the two data sets. Which one has the higher resolution?

In [ ]:
# enter your code here

We'll interpolate the chlorophyll product on to the same grid as the SST product. Let's start by plotting maps of both data sets. Enter your code below.

In [ ]:
# enter your code here

### Nearest neighbour interpolation

In nearest neighbour interpolation, we find which existing data point is nearest in space to our desired position. The value of this existing data point is then simply copied into our new array to become the value at the new point.

Let's start with a simple example again:

In [ ]:
# let's create some position values:
fake_position = np.array([0, 1.1, 2.2])
print('Original data positions: fake_position=',fake_position)
# and some observed data at those positions:
fake_data = np.array([5, 4.6, 4.1])

# and now let's imagine that we want to use a new grid
# with values at the following positions:
new_positions = np.array([0.25, 0.75, 1.25, 1.75])
# and we'll make an empty array to store the new
# data values in
new_data = np.empty(len(new_positions))

# now let's go through the steps in the nearest neighbours method:

# let's do the calculations for the first point in the new_positions array:
# 1. which point is closest to new_positions[0] ?
dx = np.abs(fake_position - new_positions[0])
print('The distance from new_positions[0]='+\
      '{} to each point in fake_position is: '.format(new_positions[0]), dx)
nearest_point = np.where(dx == np.min(dx))[0][0]
print('The nearest point to {} is index {}, with fake_position[{}]={}'.format(\
                              new_positions[0],nearest_point, nearest_point, \
                              fake_position[nearest_point]))

# 2. we copy that point into our new_data array:
new_data[0] = fake_data[nearest_point]
print('--')
print('The value at fake_data[{}] is {}'.format(nearest_point, fake_data[nearest_point]))
print('The value at new_positions[0] is now: ', new_data[0])

In [ ]:
# 3. we repeat this for each position to complete the interpolation:
print('Original data positions: fake_position=',fake_position)
print('--')
for i in range(1,len(new_positions)):
  dx = np.abs(fake_position - new_positions[i])
  print('The distance from new_positions'+\
        '[{}]={} to each point in fake_position is: '.format(i, new_positions[i]), dx)
  nearest_point = np.where(dx == np.min(dx))[0][0]
  print('The nearest point to {} is index {}, with fake_position[{}]={}'.format(\
                              new_positions[i],nearest_point, nearest_point, \
                              fake_position[nearest_point]))
  new_data[i] = fake_data[nearest_point]
  print('The value at fake_data[{}] is {}'.format(nearest_point, fake_data[nearest_point]))
  print('The value at new_positions[{}] is now: '.format(i), new_data[i])
  print('#####')

In [ ]:
# let's plot the original data and the interpolated data to see how they look:
plt.figure(figsize=(4,2))
plt.plot(fake_position,fake_data,'ko-',label='Original data')
plt.plot(new_positions,new_data,'r.-',label='Interpolated data', alpha=0.75)
plt.legend()
plt.xlabel('Positions')
plt.ylabel('Data values')
plt.grid()

Even with this simple example, we can already see that the nearest neighbours method is not necessary very precise. But it is computationally very cheap.

Luckily we don't need to perform all these calculations by hand if we want to interpolate our data. `scipy` has interpolation functions that can make this much easier for us. We'll use the `griddata` function today, and to use the nearest neighbours method, we need to specify `method=linear` when we use it.

The cell below shows how to use the `griddata` method to interpolate some artificial 2D data. We'll consider a case where our input grid is slightly rotated with respect to the output grid:

In [ ]:
# generate the fake data
fake_longitude = np.array([
    [ 0.0000,  3.5856],
    [-0.1569,  3.4288],
    [-0.3137,  3.2719]
])

fake_latitude = np.array([
    [60.0000, 60.1569],
    [60.8964, 61.0533],
    [61.7928, 61.9500]
])
fake_data = np.reshape(np.arange(6),(3,2))
# all of these arrays have size (3,2)

# generate the new grid
new_latitude_1d = np.arange(60,62,0.5)
new_longitude_1d = np.arange(0,3.6,0.5)
new_longitude, new_latitude = np.meshgrid(new_longitude_1d,
                                          new_latitude_1d)
# this generates new, 2D, regularly spaced output positions


# interpolate the data (note: there is a problem with what we
# are doing here. We'll come back to this below. Let's just do the interpolation
# for now)
from scipy.interpolate import griddata
new_data = griddata((fake_longitude.ravel(), fake_latitude.ravel()), \
                    fake_data.ravel(), (new_longitude, new_latitude), \
                    method='nearest')

# and let's plot the results
plt.figure(figsize=(8,4))
plt.subplot(121)
plt.pcolormesh(fake_longitude,fake_latitude,fake_data, vmin=0, vmax=5)
plt.title('Original data')
plt.colorbar()
xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
# add dots to show the original data positions
plt.plot(fake_longitude, fake_latitude, 'w.')

plt.subplot(122)
plt.pcolormesh(new_longitude,new_latitude,new_data, vmin=0, vmax=5)
# make the x and y limits the same as for the original data figure
# so we can compare more easily
plt.xlim(xlim)
plt.ylim(ylim)
plt.title('Interpolated data')
plt.colorbar()
# add dots to show the new data positions
plt.plot(new_longitude, new_latitude, 'w.');

Above, we interpolated the data using latitude and longitude as the x, y positions of the data points. But this is not really a good idea.

Usually, the physical distance in metres associated with 1° of longitude is not the same as the physical distance in metres associated with 1° of latitude. So if we use longitude and latitude in degrees as position data, this can lead to inaccurate results.

For all interpolation methods, this can be a problem if the data are irregularly spaced, or if one grid is rotated with respect to the other one. For regularly spaced grids, non-rotated grids, it will not have an effect for the nearest neighbour method. However, for some other interpolation methods, we can obtain different results even when interpolating from one regular, non-rotated grid to another one.

To resolve the problem, we should convert all latitudes and longitudes into distances in metres or kilometres with respect to a fixed reference point.

We can choose any point as the fixed reference point for this, because we will always be considering distances between our data points. The absolute values of the positions do not matter.

For example, if we choose the average latitude and average longitude to be our reference point, we can define x=0, y=0 at that point. We then calculate dx and dy with respect to that reference position.

A simplified way to calculate the distances is to use the equirectangular approximation. This treats the Earth's surface as a plane over local areas.It is not perfectly accurate over thousands of kilometers, but it is the standard "simple" way to convert degrees to meters for basic calculations.

To convert a change in latitude and longitude (Δlat and Δlon) into meters (
Δy and Δx), we can use these formulae:
1. For latitude (North-South distance):
$\Delta y = \Delta lat\times 111.132$ km
2. For longitude (East-West distance):
$\Delta x = \Delta lon\times 111.132 \times \cos{(reference\,\,latitude)}$ km

Note: The latitude used in the cos function must be converted from degrees to radians first.

Let's try it for our data set from the example above.

In [ ]:
# define some functions to perform the distance calculations:
def deg2rad(latitude):
  # transform degrees to radian
  return latitude*(np.pi/180)

def dx_dy(latitude,longitude,reference_latitude,reference_longitude):
  # calculate the distance between the given lat,lon point
  # and the reference lat,lon point in m
  dlat = latitude - reference_latitude
  dlon = longitude - reference_longitude
  dy = dlat*111.132*1e3
  dx = dlon*111.132*1e3*np.cos(deg2rad(reference_latitude))
  return dx, dy

##########################################################
# now let's transform our original data positions
# first we'll define our reference latitude and longitude
reference_latitude = np.mean(fake_latitude)
reference_longitude = np.mean(fake_longitude)
# then we'll use the function above to transform lon/lat to distances in m
# from that reference point:
dx, dy = dx_dy(fake_latitude,fake_longitude,reference_latitude,reference_longitude)

# now we'll also transform our new grid positions
dx_new, dy_new = dx_dy(new_latitude,new_longitude,reference_latitude,reference_longitude)

# and print out the physical distances from the central latitude, longitude in m:
print('Central position: {}°E, {}°N'.format(reference_longitude, reference_latitude))
print('Data x-positions with respect to this point, in km:', dx/1e3)
print('New grid x-positions with respect to this point, in km:', dx_new[0,:]/1e3)
print('Data y-positions with respect to this point, in km:', dy/1e3)
print('New grid y-positions with respect to this point, in km:', dy_new[:,0]/1e3)

In [ ]:
# now let's interpolate using these distances instead of latitude and longitude:
new_data_dxdy = griddata((dx.ravel(), dy.ravel()), \
                    fake_data.ravel(), (dx_new, dy_new), \
                    method='nearest')

plt.figure(figsize=(12,4))
plt.subplot(131)
plt.pcolormesh(fake_longitude,fake_latitude,fake_data, vmin=0, vmax=5)
plt.title('Original data')
plt.colorbar()
xlim = plt.gca().get_xlim()
ylim = plt.gca().get_ylim()
# add dots to show the original data positions
plt.plot(fake_longitude, fake_latitude, 'w.')

plt.subplot(132)
plt.pcolormesh(new_longitude,new_latitude,new_data, vmin=0, vmax=5)
plt.xlim(xlim)
plt.ylim(ylim)
plt.title("Interpolated data using\nlat/lon coords")
plt.colorbar()
# add dots to show the new data positions
plt.plot(new_longitude, new_latitude, 'w.')

plt.subplot(133)
plt.pcolormesh(new_longitude,new_latitude,new_data_dxdy, vmin=0, vmax=5)
plt.xlim(xlim)
plt.ylim(ylim)
plt.title("Interpolated data using\nphysical coords")
plt.colorbar()
# add dots to show the new data positions
plt.plot(new_longitude, new_latitude, 'w.');

You will now try to interpolate the chlorophyll-A data that we loaded earlier (`ds_colour`) on to the same grid as the SST (`ds_sst`) using the nearest neighbours method.

First we'll do this "wrongly", but simply, by interpolating just using longitude and latitude. Because we are using the nearest neighbours method and both of our grids are regular, this won't actually make any difference to our results in this case.

In the cell below, you will need to perform the following steps:
1. Use `np.meshgrid` to transform the latitude and longitude data for each data set into 2D arrays. You will have four 2D arrays as output.
2. Use `scipy`'s `griddata` function to interpolate the data. You can follow the example above.
5. Once you have performed the interpolation, make a plot with 2 subfigures showing the original data on one panel, and the interpolated data on the second panel. Can you see any differences in the data?

In [ ]:
# enter your code here

Now let's perform the interpolation "correctly", taking into account the physical distances.

In the cell below, you will need to perform the following steps:
1. Use `np.meshgrid` to transform the latitude and longitude data for each data set into 2D arrays. You will have four 2D arrays as output [same as step 1 above]
2. Calculate the average latitude and longitude for the `ds_colour` dataset. These will be your reference latitude and reference longitude
3. Transform the `ds_colour` latitudes and longitudes into physical distances with respect to the reference latitude and reference longitude. You can use the functions in the cells above for this.
4. Use `scipy`'s `griddata` function to interpolate the data.
5. Make plots showing your results again.

In [ ]:
# enter your code here

### Linear interpolation

Linear interpolation works by considering the two data points that surround the position where we want to estimate a value. We connect these two points with a straight line, and estimate the data value as the value given by the line at that position.

Let's consider our 1D example again:

In [ ]:
# let's create some position values:
fake_position = np.array([0, 1.1, 2.2])
print('Original data positions: fake_position=',fake_position)
# and some observed data at those positions:
fake_data = np.array([5, 4.6, 4.1])

# and now let's imagine that we want to use a new grid
# with values at the following positions:
new_positions = np.array([0.25, 0.75, 1.25, 1.75])
# and we'll make an empty array to store the new
# data values in
new_data = np.empty(len(new_positions))

# now let's go through the steps in the linear interpolation method:

# let's do the calculations for the first point in the new_positions array:
# 1. find the closest point that lies to the left of our new position:
dx = fake_position - new_positions[0] # positions to the left will be negative,
                                      # positions to the right will be positive
# which is the smallest negative value of dx?
minval = np.max(dx[dx < 0])
nearest_point_low = np.where(dx == minval)[0][0]

# which is the smallest positive value of dx?
minval = np.min(dx[dx > 0])
nearest_point_high = np.where(dx == minval)[0][0]

print('The nearest points to {} have indices {} and {},'.format(new_positions[0],
                                                                nearest_point_low, nearest_point_high)+\
                            'with fake_position['+\
                            '{}]={} and with fake_position[{}]={}'.format(\
                              nearest_point_low, fake_position[nearest_point_low],
                              nearest_point_high, fake_position[nearest_point_high]))
# 2. let's take the data values at the two positions surrounding our new point:
data_low = fake_data[nearest_point_low]
data_high = fake_data[nearest_point_high]
print('The surrounding data values are {} and {}'.format(data_low, data_high))

# 3. now let's fit a straight line
p = np.polyfit([fake_position[nearest_point_low], fake_position[nearest_point_high]], \
               [fake_data[nearest_point_low], fake_data[nearest_point_high]], 1)
plt.figure()
plt.plot(fake_position[[nearest_point_low,nearest_point_high]],
         fake_data[[nearest_point_low,nearest_point_high]],'ko-',label='Original data')

# 4. and now let's use the straight line to estimate our value at our new point:
new_data[0] = np.polyval(p, new_positions[0])
print('The value at new_positions[0] is now: ', new_data[0])
plt.plot(new_positions[0],new_data[0],'r+',markersize=11,label='Interpolated point')
plt.legend()

Let's repeat this for all the new positions:

In [ ]:
for i in range(len(new_positions)):
  # 1. find the surrounding positions for the new data position:
  dx = fake_position - new_positions[i]
  # which is the smallest negative value of dx?
  minval = np.max(dx[dx < 0])
  nearest_point_low = np.where(dx == minval)[0][0]

  # which is the smallest positive value of dx?
  minval = np.min(dx[dx > 0])
  nearest_point_high = np.where(dx == minval)[0][0]
  # 2. let's take the data values at the two positions surrounding our new point:
  data_low = fake_data[nearest_point_low]
  data_high = fake_data[nearest_point_high]

  # 3. now let's fit a straight line
  p = np.polyfit([fake_position[nearest_point_low], fake_position[nearest_point_high]], \
               [fake_data[nearest_point_low], fake_data[nearest_point_high]], 1)

  # 4. and now let's use the straight line to estimate our value at our new point:
  new_data[i] = np.polyval(p, new_positions[i])

plt.figure()
plt.plot(fake_position,fake_data,'ko-',label='Original data')
plt.plot(new_positions,new_data,'r+-',label='Interpolated data', alpha=0.75,
         markersize=10);

To use this method with real data, we can again use `scipy`'s `griddata` function. This time, we just need to specify `method='linear'` instead of `method='nearest'`

In the cell below, repeat the calculation that you did before to put the ocean colour data on the same grid as the SST, but this time use the linear interpolation method. Make sure that you give the array containing your output data a different name from the one that you used for the nearest neighbours interpolation, so that you can compare the two methods.

Plot your results again, showing maps the original and interpolated data.

In [ ]:
# enter your code here